# Experiment 040: representation numerics (Colab)

Compare adapter loaded and merged copies of the same GSM8K-SFT models on the 245 item GRPO training pool. All four scales and three seeds are fixed before the results are viewed. No training, generation or model evaluation on the MMLU test set runs here.

## 0. Compute units and Drive

In [6]:
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
#fill the starting balance before running
cu_before = 345.59
unit_rate = 6.77
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
OPEN_UNITS = os.path.join(ROOT, 'results', 'open_cu_sessions')
UNITS_LOG = os.path.join(ROOT, 'results', 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])
    if not os.path.exists(open_path):
        record = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                  'notebook': '040_colab_representation_numerics',
                  'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(record, f, indent=2)
        print('saved open session:', session_id)

session 20260820_232114 | cu_before = 345.59
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved open session: 20260820_232114


## 1. GPU check

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gib:.1f} GiB')
assert 'A100' in gpu and total_gib >= 79, 'need the A100 80GB runtime'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-fbced96f-bb28-b82d-a733-23d9194aed54)
total vram: 79.3 GiB


## 2. Repository and dependencies

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

working in: /content/INM363-IndividualProject-Dissertation/001_Code


In [4]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
print('dependencies installed')

dependencies installed


## 3. Session setup

In [7]:
import os, shutil, sys
os.chdir(code_root)
sys.path.insert(0, os.getcwd())
from config import load_config
from scripts.s40_representation import (ARMS_040, PAIR_MANIFEST, REPRESENTATIONS,
                                        SOURCE_RECORD, compare_rows, pair_dir,
                                        pair_tag, pipeline_config_path, prepare_pool, result_files,
                                        source_facts)

assert cu_before is not None and unit_rate is not None, 'fill section 0 first'
free_gib = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gib:.0f} GB')
assert free_gib >= 5, 'need at least 5GB free on Drive'
POOL = '/content/040_grpo_pool.jsonl'
print('pairs:', len(ARMS_040))

drive free: 178 GB
pairs: 12


## 4. Wiring checks

In [8]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'tests/test_040_representation.py'], check=True)
print('wiring passed')

wiring passed


## 5. Pool and source records

In [9]:
import json, os
from obs import _versions
from scripts.s12_eval import atomic_write_json, sha256
from train.model_utils import resolve_dtype

pool_record = prepare_pool(POOL)
current_versions = _versions()
sources = {}
for scale, seed in ARMS_040:
    tag = pair_tag(scale, seed)
    result_dir = pair_dir(ROOT, scale, seed)
    os.makedirs(result_dir, exist_ok=True)
    print('checking', tag)
    cfg = load_config(pipeline_config_path(scale, seed))
    resolved_dtype = resolve_dtype(cfg)
    source = source_facts(ROOT, scale, seed, pool_record, resolved_dtype)
    source_path = os.path.join(result_dir, SOURCE_RECORD)
    if os.path.exists(source_path):
        assert json.load(open(source_path)) == source, f'{tag}: source record changed'
    else:
        atomic_write_json(source_path, source)

    parent = json.load(open(source['parent_manifest']))
    # a library upgrade would change the numbers on its own, match the parent
    for key in ('torch', 'transformers', 'peft', 'accelerate', 'datasets', 'cais/mmlu@'):
        expected = parent.get('versions', {}).get(key)
        if expected is not None:
            assert current_versions.get(key) == expected, f'{tag}: {key} differs from parent'
    expected_dtype = parent.get('resolved_dtype')
    if expected_dtype is not None:
        assert resolved_dtype == expected_dtype, f'{tag}: dtype differs from parent'
    sources[(scale, seed)] = (result_dir, source_path, source)
    print(tag, 'ready')

scientific_files = ('scripts/s40_representation.py', 'scripts/s32_wsft.py',
                    'eval/harness.py', 'data/splits.py', 'train/model_utils.py')
preflight = {
    'package': '040_w19',
    'pool': pool_record,
    'versions': current_versions,
    'scientific_sha256': {path: sha256(path) for path in scientific_files},
    'source_record_sha256': {
        pair_tag(scale, seed): sha256(sources[(scale, seed)][1])
        for scale, seed in ARMS_040},
}
preflight_path = os.path.join(ROOT, 'results', '040_preflight.json')
if os.path.exists(preflight_path):
    assert json.load(open(preflight_path)) == preflight, '040 preflight changed'
else:
    atomic_write_json(preflight_path, preflight)
print('all source records committed')

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

high_school_psychology/test-00000-of-000(…): reconstructing file:   0%|          |  0.00B / 92.8kB            

high_school_psychology/test-00000-of-000(…): downloading bytes:           |  0.00B            

high_school_psychology/validation-00000-(…): reconstructing file:   0%|          |  0.00B / 15.2kB            

high_school_psychology/validation-00000-(…): downloading bytes:           |  0.00B            

... (65 outputs left out)

Generating test split:   0%|          | 0/126 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/14 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

checking 040_representation_B05_seed0
040_representation_B05_seed0 ready
checking 040_representation_B05_seed1
040_representation_B05_seed1 ready
checking 040_representation_B05_seed2
040_representation_B05_seed2 ready
checking 040_representation_B15_seed0
040_representation_B15_seed0 ready
checking 040_representation_B15_seed1
040_representation_B15_seed1 ready
checking 040_representation_B15_seed2
040_representation_B15_seed2 ready
checking 040_representation_B3_seed0
040_representation_B3_seed0 ready
checking 040_representation_B3_seed1
040_representation_B3_seed1 ready
checking 040_representation_B3_seed2
040_representation_B3_seed2 ready
checking 040_representation_B7_seed0
040_representation_B7_seed0 ready
checking 040_representation_B7_seed1
040_representation_B7_seed1 ready
checking 040_representation_B7_seed2
040_representation_B7_seed2 ready
all source records committed


## 6. Run the paired probes

In [10]:
import json, os, shutil, statistics, subprocess, sys, time
from datetime import datetime
from obs import append_ledger, gpu_snapshot
from scripts.s12_eval import (IntegrityError, atomic_write_json, read_jsonl,
                              sha256, stage_into)

LEDGER = os.path.join(ROOT, 'results', 'compute_ledger.jsonl')
assert json.load(open(preflight_path)) == preflight, 'run the preflight first'


def committed_pair(result_dir, source_path):
    manifest_path = os.path.join(result_dir, PAIR_MANIFEST)
    if not os.path.exists(manifest_path):
        return None
    manifest = json.load(open(manifest_path))
    assert manifest.get('completed') is True, f'{manifest_path}: incomplete manifest'
    assert manifest.get('source_record_sha256') == sha256(source_path),         f'{manifest_path}: source record changed'
    source = json.load(open(source_path))
    assert manifest.get('resolved_dtype') == source['resolved_dtype'],         f'{manifest_path}: dtype record changed'
    for representation in REPRESENTATIONS:
        for name in result_files(representation):
            path = os.path.join(result_dir, name)
            assert os.path.exists(path), f'{manifest_path}: missing {name}'
            assert manifest['sha256'][name] == sha256(path), f'{manifest_path}: changed {name}'
    adapter_rows = read_jsonl(os.path.join(result_dir, result_files('adapter')[1]))
    merged_rows = read_jsonl(os.path.join(result_dir, result_files('merged')[1]))
    assert manifest['comparison'] == compare_rows(adapter_rows, merged_rows)
    return manifest


for scale, seed in ARMS_040:
    tag = pair_tag(scale, seed)
    result_dir, source_path, source = sources[(scale, seed)]
    committed = committed_pair(result_dir, source_path)
    if committed:
        print(tag, 'already complete')
        continue

    names = [name for representation in REPRESENTATIONS for name in result_files(representation)]
    orphans = [name for name in names if os.path.exists(os.path.join(result_dir, name))]
    if orphans:
        archive = os.path.join(result_dir, 'orphaned_' + time.strftime('%Y%m%d_%H%M%S'))
        os.makedirs(archive, exist_ok=True)
        for name in orphans:
            shutil.move(os.path.join(result_dir, name), os.path.join(archive, name))
        print(tag, 'archived incomplete files:', orphans)

    scratch = {representation: f'/content/040_{scale}_seed{seed}_{representation}'
               for representation in REPRESENTATIONS}
    started = time.time()
    evaluated = 0
    completed = False
    try:
        for representation in REPRESENTATIONS:
            shutil.rmtree(scratch[representation], ignore_errors=True)
            process = subprocess.run([
                sys.executable, '-u', 'scripts/s40_representation.py',
                '--source-record', source_path, '--pool', POOL,
                '--representation', representation, '--out', scratch[representation],
            ])
            if process.returncode != 0:
                raise RuntimeError(f'{tag}: {representation} probe failed')
            metrics_name, preds_name = result_files(representation)
            metrics = json.load(open(os.path.join(scratch[representation], metrics_name)))
            rows = read_jsonl(os.path.join(scratch[representation], preds_name))
            assert metrics['n_items'] == 245 and len(rows) == 245
            evaluated += len(rows)

        for representation in REPRESENTATIONS:
            for name in result_files(representation):
                stage_into(result_dir, os.path.join(scratch[representation], name), name)
        adapter_rows = read_jsonl(os.path.join(result_dir, result_files('adapter')[1]))
        merged_rows = read_jsonl(os.path.join(result_dir, result_files('merged')[1]))
        comparison = compare_rows(adapter_rows, merged_rows)
        manifest = {
            'timestamp': datetime.now().isoformat(timespec='seconds'),
            'package': '040_w19', 'completed': True,
            'scale': scale, 'seed': seed,
            'source_record_sha256': sha256(source_path),
            'resolved_dtype': source['resolved_dtype'],
            'measurement_order': list(REPRESENTATIONS),
            'comparison': comparison,
            'sha256': {name: sha256(os.path.join(result_dir, name)) for name in names},
            'gpu': gpu_snapshot(), 'versions': current_versions,
            'model_test_contacts': 0,
        }
        # write the manifest after both representations are complete
        atomic_write_json(os.path.join(result_dir, PAIR_MANIFEST), manifest)
        assert committed_pair(result_dir, source_path) == manifest
        completed = True
        print(tag, comparison)
    finally:
        append_ledger(LEDGER, {
            'run_tag': tag, 'condition': 'representation_numerics',
            'timings_sec': {'pool_probe': round(time.time() - started, 1)},
            'train_sec': 0, 'train_steps': 0,
            'eval_items': {'grpo_pool': evaluated or None, 'mmlu_test': 0},
            'completed': completed, 'gpu': gpu_snapshot(),
        })
        for path in scratch.values():
            shutil.rmtree(path, ignore_errors=True)

print('all paired probes complete')

040_representation_B05_seed0 {'n_items': 245, 'agreement': 0.95102041, 'n_changed': 12, 'changed_fraction': 0.04897959, 'adapter_accuracy': 0.5755102, 'merged_accuracy': 0.56734694, 'accuracy_delta': -0.00816327, 'flips': {'correct_to_wrong': 4, 'wrong_to_correct': 2, 'wrong_to_different_wrong': 6}, 'top_two_margin': {'changed_mean': 0.0625, 'changed_median': 0.0625, 'unchanged_mean': 1.19380365, 'unchanged_median': 0.75}, 'changed_by_subject': {'formal_logic': 4, 'high_school_mathematics': 6, 'high_school_psychology': 1, 'philosophy': 1}}
040_representation_B05_seed1 {'n_items': 245, 'agreement': 0.96734694, 'n_changed': 8, 'changed_fraction': 0.03265306, 'adapter_accuracy': 0.57142857, 'merged_accuracy': 0.5755102, 'accuracy_delta': 0.00408163, 'flips': {'correct_to_wrong': 3, 'wrong_to_correct': 4, 'wrong_to_different_wrong': 1}, 'top_two_margin': {'changed_mean': 0.0546875, 'changed_median': 0.0, 'unchanged_mean': 1.17919304, 'unchanged_median': 0.75}, 'changed_by_subject': {'forma

## 7. Summary

In [11]:
import json, os, statistics
from scripts.s12_eval import IntegrityError, atomic_write_json, sha256

by_scale = {}
pair_hashes = {}
for scale in ('B05', 'B15', 'B3', 'B7'):
    rows = []
    for seed in (0, 1, 2):
        result_dir, source_path, _ = sources[(scale, seed)]
        manifest = committed_pair(result_dir, source_path)
        assert manifest is not None, f'{pair_tag(scale, seed)} is not complete'
        rows.append(manifest['comparison'])
        pair_path = os.path.join(result_dir, PAIR_MANIFEST)
        pair_hashes[pair_tag(scale, seed)] = sha256(pair_path)
    agreements = [row['agreement'] for row in rows]
    changes = [row['changed_fraction'] for row in rows]
    deltas = [row['accuracy_delta'] for row in rows]
    by_scale[scale] = {
        'agreement_mean': round(statistics.mean(agreements), 8),
        'agreement_range': [min(agreements), max(agreements)],
        'changed_fraction_mean': round(statistics.mean(changes), 8),
        'changed_fraction_range': [min(changes), max(changes)],
        'accuracy_delta_mean': round(statistics.mean(deltas), 8),
        'accuracy_delta_range': [min(deltas), max(deltas)],
    }

summary = {
    'package': '040_w19', 'completed': True, 'n_pairs': 12,
    'model_test_contacts': 0, 'by_scale': by_scale,
        # bind the summary to the exact pair manifests it was built from
    'pair_manifest_sha256': pair_hashes,
}
summary_path = os.path.join(ROOT, 'results', '040_representation_summary.json')
if os.path.exists(summary_path):
    if json.load(open(summary_path)) != summary:
        raise IntegrityError(f'{summary_path}: summary changed')
else:
    atomic_write_json(summary_path, summary)

for scale, values in by_scale.items():
    print(scale, '| agreement', f"{100 * values['agreement_mean']:.2f}%",
          '| changed', f"{100 * values['changed_fraction_mean']:.2f}%",
          '| accuracy delta', f"{100 * values['accuracy_delta_mean']:+.2f}pt")

B05 | agreement 95.37% | changed 4.63% | accuracy delta +0.41pt
B15 | agreement 97.96% | changed 2.04% | accuracy delta +0.00pt
B3 | agreement 97.55% | changed 2.45% | accuracy delta -0.41pt
B7 | agreement 98.78% | changed 1.22% | accuracy delta +0.14pt


## 8. Close the units record

In [13]:
import json, os, time
cu_after = 336.59
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif not os.path.exists(open_path):
    print('no open record for this session')
else:
    with open(open_path) as f:
        record = json.load(f)
    record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
    record['colab_cu_after'] = cu_after
    record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
    record['closed_on_resume'] = False
    with open(UNITS_LOG, 'a') as f:
        f.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('logged:', record)

logged: {'session_id': '20260820_232114', 'ts_start': '2026-08-20 23:42', 'notebook': '040_colab_representation_numerics', 'colab_cu_before': 345.59, 'unit_rate': 6.77, 'ts_end': '2026-08-21 00:42', 'colab_cu_after': 336.59, 'colab_cu_used': 9.0, 'closed_on_resume': False}
